In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
from src.benchmark.build import build_benchmark
from src.benchmark.evaluate import (StyleSpace, evaluate, summarize,
                                    learning_curve, DEFAULT_FEATURES)

df = pd.read_csv(ROOT / "data" / "processed" / "enron_messages.csv")
df["message_id"] = df["message_id"].astype(str)
FEATURES = [f for f in DEFAULT_FEATURES if (df[f] != 0).mean() >= 0.01]
print("features used:", FEATURES)
print("dropped (almost always zero):", [f for f in DEFAULT_FEATURES if f not in FEATURES])
print(df["user_split"].value_counts().to_dict(), df["time_split"].value_counts().to_dict())

bench = build_benchmark(df, n_neg=9, eval_splits=("val", "test"), max_items_per_user=50, seed=0)
bench.to_csv(ROOT / "data" / "processed" / "benchmark_v0.csv", index=False)
n_items = bench["item_id"].nunique()
print(f"{n_items} items from {bench['user_id'].nunique()} users, {len(bench)} lineups")
if n_items < 200:
    print("WARNING: few items -> noisy results. Consider eval_splits=('train','val','test') "
          "for the untrained baselines, or a lower MIN_MSGS in notebook 03.")

by_id = df.set_index("message_id")
ex = bench[bench.tier == "content_and_length"].sample(3, random_state=1)
for item, neg in zip(ex["item_id"], ex["neg_ids"]):
    print("REAL    :", by_id.loc[item, "message"][:160].replace("\n", " "))
    for n in neg.split(";")[:3]:
        print("  other :", by_id.loc[n, "message"][:160].replace("\n", " "))
    print()

space = StyleSpace(df, FEATURES)
res = evaluate(space, bench, seed=0)
summ = summarize(res)
summ.round(3)

lc = learning_curve(space, bench)
lc.round(3)

features used: ['log_words', 'avg_sent_len', 'punct_density', 'n_questions', 'n_exclam', 'caps_ratio', 'starts_lower', 'has_newline']
dropped (almost always zero): []
{'train': 43158, 'val': 12629, 'test': 11918} {'history': 54201, 'eval': 13504}
1179 items from 29 users, 4716 lineups
REAL    : Oh, Terrie, Ken won't be in town.  Otherwise, I'm sure he'd love to.  TERRIE JAMES@ENRON COMMUNICATIONS 04/16/2001 10:26 AM To: Kenneth Lay/Corp/Enron@ENRON, Je
  other : I think we should do this but should include some comments praising FERC for taking some positive actions and really looking for solutions instead of simply way
  other : No. We can wait until he gets back on the 1st.  Thanks.  Carol St. Clair EB 3889 713-853-3989 (Phone) 713-646-3393 (Fax) carol.st.clair@enron.com  Suzanne Adams
  other : I'm remembering now.  It was a term sheet which was never executed, as far as I know.  I believe FPL prepared it, and we may have commented on it. Let me see wh

REAL    : Audrey,  Please sen

tier,random,length_matched,content_similar,content_and_length
history_msgs,,,,
0,0.100,0.100,0.100,0.100
1,0.184,0.154,0.157,0.145
3,0.214,0.193,0.177,0.158
10,0.254,0.221,0.201,0.181
30,0.291,0.248,0.225,0.204
100,0.293,0.257,0.218,0.205
all,0.306,0.247,0.227,0.216
